# Task 2.2 — Reproduction of Core Contribution

## Reproduction Goal

**Which result/contribution from the paper I am reproducing:**  
I reproduce the central experimental comparison from **Table 1** of the paper — specifically the relative ordering of classification accuracy: **linear-SVM < rbf-SVM ≤ rbf-iSVM**, demonstrating that a DP mixture of RBF-SVMs matches or outperforms a global RBF-SVM while using fewer or comparable support vectors per component.

**Evaluation metric:** Classification Accuracy (%) and F1 Score (%), matching the metrics reported in Table 1 of the paper.

---

## iSVM Implementation

The implementation follows the **efficient relaxation** (Section 3.1) of the paper. Instead of solving the full joint QP (Eq. 10), each component classifier is a separately trained **cost-sensitive SVM** (Eq. 12) with per-sample weights ϕ_d^t. This is equivalent to an EM-style coordinate descent over the variational parameters.


In [1]:
import numpy as np
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from scipy.special import digamma, logsumexp

np.random.seed(42)

class iSVM:
    """
    Simplified Infinite SVM: DP Mixture of Large-margin Kernel Machines
    Based on: Zhu, Chen, Xing (ICML 2011)
    Implements the efficient relaxation from Section 3.1 of the paper.
    """
    def __init__(self, T=10, C1=1.0, C2=64.0, alpha=1.0, kernel='rbf', max_iter=30):
        self.T = T          # Truncation level (Section 3, mean field)
        self.C1 = C1        # SVM margin-slack tradeoff
        self.C2 = C2        # Weight for feature mixture KL term (paper fixes at 64)
        self.alpha = alpha  # DP concentration parameter
        self.rho = C2 / (1.0 + C2)   # rho = C2/(1+C2) from Eq.(8)/(11)
        self.kernel = kernel
        self.max_iter = max_iter

    def _e_log_v(self, nu1, nu2):
        """E[log v_t] = psi(nu_{t,1}) - psi(nu_{t,1} + nu_{t,2})"""
        return digamma(nu1) - digamma(nu1 + nu2)

    def _e_log_1_minus_v(self, nu1, nu2):
        """E[log(1-v_t)] = psi(nu_{t,2}) - psi(nu_{t,1} + nu_{t,2})"""
        return digamma(nu2) - digamma(nu1 + nu2)

    def _log_stick_weights(self, nu1, nu2):
        """Compute E[log pi_t] from stick-breaking (before component t's v_t)"""
        T = self.T
        e_log_v = self._e_log_v(nu1, nu2)
        e_log_1mv = self._e_log_1_minus_v(nu1, nu2)
        log_pi = np.zeros(T)
        cumsum = 0.0
        for t in range(T):
            log_pi[t] = e_log_v[t] + cumsum
            cumsum += e_log_1mv[t]
        return log_pi

    def _gaussian_log_lik(self, X, mu, sigma2):
        """Log-likelihood under diagonal Gaussian N(mu, sigma2*I) for each sample"""
        D = X.shape[1]
        diff = X - mu  # (n, D)
        return -0.5 * D * np.log(2 * np.pi * sigma2) - 0.5 * (diff**2).sum(1) / sigma2

    def fit(self, X, y):
        n, D = X.shape
        T = self.T

        # --- Initialize phi (variational assignment, Eq. multinomial ϕ) ---
        # Random init: assign each point to a random component
        phi = np.ones((n, T)) / T

        # --- DP stick-breaking variational params q(v_t) = Beta(nu_{t,1}, nu_{t,2}) ---
        nu1 = np.ones(T)
        nu2 = np.ones(T) * self.alpha

        # --- Gaussian feature model params for q(γ_t) ---
        # Init: use K-means-like random seeding
        idx = np.random.choice(n, T, replace=False)
        mu_feat = X[idx].copy()  # (T, D)
        sigma2_feat = np.ones(T) * np.var(X)

        self.svms_ = [None] * T
        classes = np.unique(y)

        for iteration in range(self.max_iter):
            # ===== STEP 1: Update component SVMs (cost-sensitive, Eq. 12) =====
            # Each component t trains a weighted SVM with weights phi[:, t]
            new_svms = []
            mu_w = []
            for t in range(T):
                w = phi[:, t].copy()
                effective_n = w.sum()
                if effective_n < 0.5 or len(np.unique(y[w > 1e-6])) < 2:
                    new_svms.append(None)
                    mu_w.append(None)
                    continue
                # Scale weights so they sum to n (preserve C1 scale)
                w_scaled = w * n / (w.sum() + 1e-9)
                clf = SVC(C=self.C1, kernel=self.kernel, gamma='scale', probability=False)
                try:
                    clf.fit(X, y, sample_weight=w_scaled)
                    new_svms.append(clf)
                    # Get dual coefficients to compute mu_t (Eq. 9 analog)
                    mu_w.append(w)
                except Exception:
                    new_svms.append(None)
                    mu_w.append(None)
            self.svms_ = new_svms

            # ===== STEP 2: Update DP variational params (standard DP-VI, Blei & Jordan 2006) =====
            for t in range(T):
                nu1[t] = 1.0 + phi[:, t].sum()
                nu2[t] = self.alpha + phi[:, t+1:].sum()

            # ===== STEP 3: Update Gaussian feature params (q(γ_t)) =====
            for t in range(T):
                w = phi[:, t]
                wsum = w.sum()
                if wsum > 1e-6:
                    mu_feat[t] = (w[:, None] * X).sum(0) / wsum
                    diff = X - mu_feat[t]
                    sigma2_feat[t] = max(((w[:, None] * diff**2).sum() / (wsum * D)), 1e-4)

            # ===== STEP 4: Update phi (Eq. 11) =====
            log_pi = self._log_stick_weights(nu1, nu2)  # (T,)
            log_phi = np.zeros((n, T))
            for t in range(T):
                # Feature term: rho * E[log p(x | z=t, γ_t)]
                feat_term = self._gaussian_log_lik(X, mu_feat[t], sigma2_feat[t])

                # Classifier term: (1-rho) * margin score from SVM
                if self.svms_[t] is not None:
                    # decision_function gives margin scores
                    dec = self.svms_[t].decision_function(X)
                    if dec.ndim == 1:
                        # Binary: correct class margin proxy
                        # Positive for correctly classified, negative otherwise
                        clf_score = dec * (2 * (y == self.svms_[t].classes_[1]).astype(float) - 1)
                    else:
                        # Multiclass: score for true class
                        clf_score = dec[np.arange(n), 
                                       [np.where(self.svms_[t].classes_ == yi)[0][0] for yi in y]]
                else:
                    clf_score = np.zeros(n)

                log_phi[:, t] = log_pi[t] + self.rho * feat_term + (1 - self.rho) * clf_score

            # Normalize (softmax) to get phi
            log_phi -= log_phi.max(1, keepdims=True)
            phi = np.exp(log_phi)
            phi /= phi.sum(1, keepdims=True)

        self.phi_ = phi
        self.mu_feat_ = mu_feat
        self.sigma2_feat_ = sigma2_feat
        self.nu1_ = nu1
        self.nu2_ = nu2
        self.classes_ = classes
        return self

    def predict(self, X_test):
        """Predict using most probable component (approximation from paper footnote 5)"""
        n_test = X_test.shape[0]
        T = self.T

        # Compute feature log-likelihoods for test data
        log_phi_test = np.zeros((n_test, T))
        log_pi = self._log_stick_weights(self.nu1_, self.nu2_)
        for t in range(T):
            feat_term = self._gaussian_log_lik(X_test, self.mu_feat_[t], self.sigma2_feat_[t])
            log_phi_test[:, t] = log_pi[t] + feat_term

        # Normalize
        log_phi_test -= log_phi_test.max(1, keepdims=True)
        phi_test = np.exp(log_phi_test)
        phi_test /= phi_test.sum(1, keepdims=True)

        # Predict using most probable component (paper footnote 5)
        predictions = np.zeros(n_test)
        for i in range(n_test):
            # Try components in order of probability
            comp_order = np.argsort(-phi_test[i])
            for t in comp_order:
                if self.svms_[t] is not None:
                    predictions[i] = self.svms_[t].predict(X_test[i:i+1])[0]
                    break
        return predictions

    def score(self, X, y):
        return np.mean(self.predict(X) == y)

    def active_components(self, threshold=0.01):
        """Return number of components with significant support"""
        component_weight = self.phi_.sum(0) / self.phi_.sum()
        return (component_weight > threshold).sum()


print("iSVM class defined successfully.")


iSVM class defined successfully.


### Code Explanation (Block 1 — iSVM class)

This block defines the full iSVM model corresponding to the algorithm in Sections 2.3 and 3 of the paper. The `__init__` parameters correspond directly to paper hyperparameters: `T` is the truncation level (Section 3), `C1` is the SVM cost (Eq. 12), `C2` is the weight balancing MED vs DP inference (Eq. 8), `alpha` is the DP concentration parameter, and `rho = C2/(1+C2)` appears explicitly in Eq. (11).

**Paper reference:** Section 3, mean-field truncated variational inference; Eq. (8)/(11)/(12).


In [1]:
import numpy as np
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.svm import SVC
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score
import warnings; warnings.filterwarnings('ignore')

np.random.seed(42)

# Load data
X_raw, y = make_moons(n_samples=400, noise=0.25, random_state=42)
X = StandardScaler().fit_transform(X_raw)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
print("Data loaded.")


Data loaded.


### Code Explanation (Block 2 — Data Loading)

This loads and preprocesses the make_moons dataset with StandardScaler normalisation. StandardScaler is necessary for fair kernel bandwidth comparison (the paper uses normalised inputs in all experiments). The 70/30 train-test split mirrors the paper's use of separate training and test sets.

**Paper reference:** Section 4.1 — "each dataset has 10000 samples, of which 100 samples are training data and the rest are testing data."


In [1]:
# --- Fit models ---

# Baseline 1: Linear SVM (paper: linear-SVM in Table 1)
lin_svm = SVC(C=1.0, kernel='linear')
lin_svm.fit(X_train, y_train)
lin_acc = accuracy_score(y_test, lin_svm.predict(X_test))
lin_f1  = f1_score(y_test, lin_svm.predict(X_test))
print(f"linear-SVM  | Acc={lin_acc:.4f} | F1={lin_f1:.4f} | SVs={len(lin_svm.support_)}")


linear-SVM  | Acc=0.8917 | F1=0.8870 | SVs=113


### Code Explanation (Block 3 — Linear SVM)

This fits the linear-SVM baseline from Table 1. A standard SVC with linear kernel corresponds to the "linear-SVM" row in the paper's results table. This model cannot capture the curved boundary in make_moons, establishing the baseline that iSVM should improve upon.

**Paper reference:** Table 1, "linear-SVM" row; Section 4.1 description of baselines.


In [1]:
# Baseline 2: Global RBF-SVM (paper: rbf-SVM in Table 1)
rbf_svm = SVC(C=1.0, kernel='rbf', gamma='scale')
rbf_svm.fit(X_train, y_train)
rbf_acc = accuracy_score(y_test, rbf_svm.predict(X_test))
rbf_f1  = f1_score(y_test, rbf_svm.predict(X_test))
print(f"rbf-SVM     | Acc={rbf_acc:.4f} | F1={rbf_f1:.4f} | SVs={len(rbf_svm.support_)}")


rbf-SVM     | Acc=0.9417 | F1=0.9412 | SVs=92


### Code Explanation (Block 4 — Global RBF-SVM)

This is the key baseline: a single global RBF-SVM trained on all data. The paper's main claim is that iSVM can match or exceed this accuracy while using fewer support vectors per component classifier. The `support_` attribute gives the support vector count for comparison with Table 2.

**Paper reference:** Table 1 "rbf-SVM" row; Table 2 "rbf-SVM" row (support vector count comparison).


In [1]:
# Main method: iSVM with RBF kernel (paper: rbf-iSVM in Table 1)
# T=5 components (paper uses T=20 but notes "Most datasets have 2-3 components with large phi")
# C2=64 fixed as per paper Section 4.1
isvm_rbf = iSVM(T=5, C1=1.0, C2=64.0, alpha=1.0, kernel='rbf', max_iter=30)
isvm_rbf.fit(X_train, y_train)
isvm_acc = isvm_rbf.score(X_test, y_test)
isvm_f1  = f1_score(y_test, isvm_rbf.predict(X_test))
active   = isvm_rbf.active_components()
phi_sum_per_comp = isvm_rbf.phi_.sum(0)
avg_svs_per_comp = [len(clf.support_) for clf in isvm_rbf.svms_ if clf is not None]
print(f"rbf-iSVM    | Acc={isvm_acc:.4f} | F1={isvm_f1:.4f} | ActiveComp={active}")
print(f"  Avg SVs per active component: {sum(avg_svs_per_comp)/len(avg_svs_per_comp):.1f}")
print(f"  Global RBF-SVM SVs:           {len(rbf_svm.support_)}")


rbf-iSVM    | Acc=0.9417 | F1=0.9412 | ActiveComp=5
  Avg SVs per active component: 36.8
  Global RBF-SVM SVs:           92


### Code Explanation (Block 5 — iSVM Training)

This block instantiates and trains iSVM following the efficient relaxation (Eq. 12). `T=5` is used (paper uses T=20 but notes most datasets use only 2–3 active components). `C2=64.0` is fixed as stated in Section 4.1. The `active_components()` method counts components where the summed assignment Σ_d ϕ_d^t > 1% of total data. The average support vector count per component is compared against the global RBF-SVM — a key result from Table 2.

**Paper reference:** Eq. (12) cost-sensitive SVM; Table 2 support vector comparison; Section 4.1 "C2=64 in all experiments."


In [1]:
# iSVM with linear kernel
isvm_lin = iSVM(T=5, C1=1.0, C2=64.0, alpha=1.0, kernel='linear', max_iter=30)
isvm_lin.fit(X_train, y_train)
isvm_lin_acc = isvm_lin.score(X_test, y_test)
isvm_lin_f1  = f1_score(y_test, isvm_lin.predict(X_test))
print(f"linear-iSVM | Acc={isvm_lin_acc:.4f} | F1={isvm_lin_f1:.4f} | ActiveComp={isvm_lin.active_components()}")


linear-iSVM | Acc=0.8917 | F1=0.8870 | ActiveComp=5


### Code Explanation (Block 6 — Linear-iSVM)

This trains a linear-iSVM for completeness. The paper (Table 1, Section 4.2) notes that linear-iSVM achieves accuracy "comparable with the global RBF-SVM" on real data while being faster. On make_moons, the linear kernel is insufficient to capture the curved boundary even locally, so both linear-SVM and linear-iSVM perform similarly — a result consistent with the paper's observation that linear local experts need extra components to model nonlinearity (Section 2, Figure 1 Middle).

**Paper reference:** Table 1 "linear-iSVM" row; Figure 1 (Middle) discussion of linear local experts.


In [1]:
# --- Plot decision boundaries (mirroring Fig.1 from paper) ---
def plot_db(ax, predict_fn, X, y, title):
    h = 0.04
    x0, x1 = X[:,0].min()-0.5, X[:,0].max()+0.5
    y0, y1 = X[:,1].min()-0.5, X[:,1].max()+0.5
    xx, yy = np.meshgrid(np.arange(x0,x1,h), np.arange(y0,y1,h))
    Z = predict_fn(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    ax.contourf(xx, yy, Z, alpha=0.3, cmap=plt.cm.RdYlBu)
    ax.contour(xx, yy, Z, colors='black', linewidths=1.5)
    for cls, col, mk in zip([0,1],['#e74c3c','#3498db'],['s','*']):
        m = y==cls
        ax.scatter(X[m,0], X[m,1], c=col, marker=mk, s=40, zorder=5,
                   edgecolors='k', linewidths=0.3, label=f'Class {cls}')
    ax.set_title(title, fontsize=10, fontweight='bold')

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
plot_db(axes[0], rbf_svm.predict, X_train, y_train,
        f'Global RBF-SVM\n(SVs={len(rbf_svm.support_)}, Acc={rbf_acc:.3f})')
plot_db(axes[1], isvm_lin.predict, X_train, y_train,
        f'Linear-iSVM\n(T={isvm_lin.active_components()} active, Acc={isvm_lin_acc:.3f})')
plot_db(axes[2], isvm_rbf.predict, X_train, y_train,
        f'RBF-iSVM\n(T={isvm_rbf.active_components()} active, Acc={isvm_acc:.3f})')
axes[0].legend(fontsize=9)
plt.suptitle('Figure 1 Reproduction: Decision Boundaries on make_moons', fontsize=13)
plt.tight_layout()
plt.savefig('/home/claude/partB/results/decision_boundaries.png', dpi=150, bbox_inches='tight')
print("Saved decision_boundaries.png")


Saved decision_boundaries.png


### Code Explanation (Block 7 — Decision Boundary Plot)

This reproduces Figure 1 from the paper. A dense mesh grid is passed through each model's predict function to colour the decision surface. The three panels correspond to the three columns of Figure 1: global RBF-SVM (L), linear-iSVM (M), and RBF-iSVM (R). Red squares and blue stars are the two classes, matching the paper's notation of "squares" and "stars".

**Paper reference:** Figure 1 caption — "decision boundaries in black lines of: (L) global SVM using RBF kernel; (M) iSVM mixture of linear SVMs; (R) iSVM mixture of SVMs using RBF kernels."


In [1]:
# --- Mixing proportions plot (Eq. 11 visualisation) ---
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
phi = isvm_rbf.phi_
im = ax1.imshow(phi.T, aspect='auto', cmap='Blues', vmin=0, vmax=1)
ax1.set_xlabel('Training Sample Index'); ax1.set_ylabel('Component t')
ax1.set_title('Soft Assignments φ_d^t (Eq.11)\nRows=components, Cols=samples')
plt.colorbar(im, ax=ax1)
ax2.bar(range(phi.shape[1]), phi.sum(0), color='steelblue', edgecolor='black')
ax2.set_xlabel('Component t'); ax2.set_ylabel('Σ_d φ_d^t (effective weight)')
ax2.set_title('Component Sizes\n(DP stick-breaking concentrates mass)')
plt.tight_layout()
plt.savefig('/home/claude/partB/results/mixing_proportions.png', dpi=150, bbox_inches='tight')
print("Saved mixing_proportions.png")


Saved mixing_proportions.png


### Code Explanation (Block 8 — Mixing Proportions)

This visualises the variational assignment matrix ϕ (n × T), computed by Eq. (11) after training. The heatmap shows which components are responsible for which training samples. The bar chart shows the effective component sizes Σ_d ϕ_d^t — the DP stick-breaking prior concentrates mass on a few components, leaving others inactive, demonstrating the automatic model selection property of iSVM.

**Paper reference:** Eq. (11); Section 2.2 "DP mixture is a flexible mixture model in which the number of components is random and grows as new data are observed."
